In [ ]:
import json
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon

### Significance Test 1: Selectiveness

**"For each censoring method (feature noise, label noise, omission), does sensitive performance degrade significantly more than non-sensitive performance at each noise level?"**

- Null hypothesis (H₀): Sensitive and non-sensitive regions degrade equally (no selective effect)
- Alternative hypothesis (H₁): Sensitive region degrades more than non-sensitive region (selective censoring works)

For each method, at each noise level, where the baseline is unmodified, raw data:
- sensitive_degradation = (baseline_sensitive - noisy_sensitive) / baseline_sensitive * 100
- non_sensitive_degradation = (baseline_non_sensitive - noisy_non_sensitive) / baseline_non_sensitive * 100

Test: sensitive_degradation > non_sensitive_degradation

In [ ]:
def local_spearman(y, yhat, threshold, above=True):
    # Filter y and yhat based on the threshold
    if above:
        mask = np.array(y) > threshold
    else:
        mask = np.array(y) <= threshold

    local_y = np.array(y)[mask]
    local_yhat = np.array(yhat)[mask]

    if len(local_y) > 1:  # Ensure there are at least 2 data points
        corr, _ = spearmanr(local_y, local_yhat)
        return corr
    else:
        return np.nan  # Not enough data points for a valid correlation

In [ ]:
def test_selective_censoring_mlp(history_file, censor_type, censor_intervals, 
                                  censor_region, metric='corr'):
    with open(history_file, 'r') as f:
        history_data = json.load(f)
    
    all_trials_results = history_data

    # infer task keys based on censor_type
    if censor_type == 'omit':
        tasks = [f'omit {int(round(f*100))}%' for f in censor_intervals]
    elif censor_type == 'xnoise':
        tasks = [f'xn_level{f:.1f}' for f in censor_intervals]
    elif censor_type == 'ynoise':
        tasks = [f'y noise level {f}' for f in censor_intervals]
    else:
        raise ValueError(f"Unknown censor_type: {censor_type}. Must be 'omit', 'xnoise', or 'ynoise'")

    # compute correlations per trial per task
    corr_above = []  # shape: (n_trials, n_tasks)
    corr_below = []
    
    for result in all_trials_results:
        threshold = result['censor_threshold']
        ytest = result['y_test']
        row_above = []
        row_below = []
        for task in tasks:
            yhat = result['pred'][task]
            row_above.append(local_spearman(ytest, yhat, threshold, above=True))
            row_below.append(local_spearman(ytest, yhat, threshold, above=False))
        corr_above.append(row_above)
        corr_below.append(row_below)
    
    all_upper = np.array(corr_above)  # shape: (n_trials, n_tasks)
    all_lower = np.array(corr_below)

    if censor_region == 'above':
        sensitive_data = all_upper
        non_sensitive_data = all_lower
    elif censor_region == 'below':
        sensitive_data = all_lower
        non_sensitive_data = all_upper
    else:
        raise ValueError("censor_region must be 'above' or 'below'")

    baseline_sensitive = sensitive_data[:, 0]
    baseline_non_sensitive = non_sensitive_data[:, 0]

    results = []

    for censor_idx in range(1, len(tasks)):
        sens_deg = (baseline_sensitive - sensitive_data[:, censor_idx]) / np.abs(baseline_sensitive) * 100
        non_sens_deg = (baseline_non_sensitive - non_sensitive_data[:, censor_idx]) / np.abs(baseline_non_sensitive) * 100

        sens_corr_avg = np.mean(sensitive_data[:, censor_idx])
        non_sens_corr_avg = np.mean(non_sensitive_data[:, censor_idx])
        baseline_sens_avg = np.mean(baseline_sensitive)
        baseline_non_sens_avg = np.mean(baseline_non_sensitive)

        try:
            statistic, p_value = wilcoxon(sens_deg, non_sens_deg, alternative='greater')
        except ValueError:
            p_value = 1.0

        sig_marker = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''

        results.append({
            'Method': censor_type,
            'Noise_Level': censor_intervals[censor_idx],
            'Baseline_Sens': f"{baseline_sens_avg:.3f}",
            'Current_Sens': f"{sens_corr_avg:.3f}",
            'Baseline_NonSens': f"{baseline_non_sens_avg:.3f}",
            'Current_NonSens': f"{non_sens_corr_avg:.3f}",
            'p_value': p_value,
            'p_formatted': f"{p_value:.3f}{sig_marker}",
            'significant': p_value < 0.05
        })

    return pd.DataFrame(results)

In [ ]:
censor_region = 'above'

In [ ]:
# fix history.json as needed -- to make it valid JSON
root_dir = "./all_results/"

for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename == "history.json":
            filepath = os.path.join(dirpath, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                raw = f.read()

            start = raw.find("[")
            if start == -1:
                start = raw.find("{")
            if start == -1:
                print(f"SKIP (no JSON): {filepath}")
                continue

            cleaned = json.loads(raw[start:])

            with open(filepath, "w", encoding="utf-8") as f:
                json.dump(cleaned, f, indent=4)

            print(f"Fixed: {filepath}")

In [ ]:
# Load the 10% sensitive data omission experiment history
history_file = f'./all_results/mlp_omit_results_split0.1_{censor_region}/history.json'
censor_type = 'omit'
omit_fractions = np.linspace(0, 1, int(1/0.1+1))

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    omit_fractions,
    censor_region=censor_region
)
df

In [ ]:
# Load the 50% sensitive data omission experiment history
history_file = f'./all_results/mlp_omit_results_split0.5_{censor_region}/history.json'
censor_type = 'omit'
omit_fractions = np.linspace(0, 1, int(1/0.1+1))

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    omit_fractions,
    censor_region=censor_region
)
df

In [ ]:
# Load the 90% sensitive data omission experiment history
history_file = f'./all_results/mlp_omit_results_split0.9_{censor_region}/history.json'
censor_type = 'omit'
omit_fractions = np.linspace(0, 1, int(1/0.1+1))

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    omit_fractions,
    censor_region=censor_region
)
df

In [ ]:
x_noise_levels = np.linspace(0, 2, int(2/0.1+1))

In [ ]:
# Load 10% x-noise experiment history
history_file = f'./all_results/mlp_xnoise_results_split0.1_{censor_region}/history.json'
censor_type = 'xnoise'

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    x_noise_levels,
    censor_region=censor_region
)
df

In [ ]:
# Load 50% x-noise experiment history
history_file = f'./all_results/mlp_xnoise_results_split0.5_{censor_region}/history.json'
censor_type = 'xnoise'

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    x_noise_levels,
    censor_region=censor_region
)
df

In [ ]:
# Load 90% x-noise experiment history
history_file = f'./all_results/mlp_xnoise_results_split0.9_{censor_region}/history.json'
censor_type = 'xnoise'

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    x_noise_levels,
    censor_region=censor_region
)
df

In [ ]:
y_noise_levels = np.linspace(0, 10, int(2/0.2+1))

In [ ]:
# Load 10% y-noise experiment history
history_file = f'./all_results/mlp_ynoise_results_split0.1_{censor_region}/history.json'
censor_type = 'ynoise'

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    y_noise_levels,
    censor_region=censor_region
)
df

In [ ]:
# Load 50% y-noise experiment history
history_file = f'./all_results/mlp_ynoise_results_split0.5_{censor_region}/history.json'
censor_type = 'ynoise'

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    y_noise_levels,
    censor_region=censor_region
)
df

In [ ]:
# Load 90% y-noise experiment history
history_file = f'./all_results/mlp_ynoise_results_split0.9_{censor_region}/history.json'
censor_type = 'ynoise'

df = test_selective_censoring_mlp(
    history_file, 
    censor_type, 
    y_noise_levels,
    censor_region=censor_region
)
df

### Significance Test 2: Effectiveness
**"Between censoring methods, which method most effectively degrades sensitive region performance?"**

- Null hypothesis (H₀): All methods degrade sensitive performance equally
- Alternative hypothesis (H₁): One method degrades sensitive performance significantly more than another

For pairwise comparisons between methods at maximum noise level:

- method1_sensitive_degradation = (baseline_sensitive - max_noise_sensitive) / baseline_sensitive * 100
- method2_sensitive_degradation = (baseline_sensitive - max_noise_sensitive) / baseline_sensitive * 100

Test: method1_sensitive_degradation > method2_sensitive_degradation

In [ ]:
def test_method_effectiveness_mlp(history_files, method_names, censor_region, censor_split='0.1'):
    """
    Test 2: Compare method effectiveness across all 11 matched intensity levels.
    
    Matched intervals:
    - omit:   np.linspace(0, 1, 11)   → 11 levels
    - xnoise: np.linspace(0, 2, 21)   → every 2nd = 11 levels (0, 0.2, 0.4, ..., 2.0)
    - ynoise: np.linspace(0, 10, 11)  → 11 levels
    """
    
    # define intervals per method
    intervals = {
        'Feature Noise': np.linspace(0, 2, int(2/0.1+1)),   # 21 levels, subsample to 11
        'Label Noise':   np.linspace(0, 10, int(2/0.2+1)),  # 11 levels
        'Omission':      np.linspace(0, 1, int(1/0.1+1)),   # 11 levels
    }
    
    # subsampled indices for xnoise to get 11 matched levels
    xnoise_indices = list(range(0, 21, 2))  # [0, 2, 4, ..., 20]
    
    # load and compute correlations for each method
    method_sensitive = {}   # method_name -> (n_trials, n_levels) array
    
    for hist_file, method_name in zip(history_files, method_names):
        with open(hist_file, 'r') as f:
            history_data = json.load(f)
        
        all_trials_results = history_data
        
        # get task keys
        censor_type = {'Feature Noise': 'xnoise', 
                       'Label Noise': 'ynoise', 
                       'Omission': 'omit'}[method_name]
        
        censor_intervals = intervals[method_name]
        
        if censor_type == 'omit':
            tasks = [f'omit {int(round(f*100))}%' for f in censor_intervals]
        elif censor_type == 'xnoise':
            tasks = [f'xn_level{f:.1f}' for f in censor_intervals]
        elif censor_type == 'ynoise':
            tasks = [f'y noise level {f}' for f in censor_intervals]
        
        # compute correlations per trial per task
        corr_above = []
        corr_below = []
        
        for result in all_trials_results:
            threshold = result['censor_threshold']
            ytest = result['y_test']
            row_above = []
            row_below = []
            for task in tasks:
                yhat = result['pred'][task]
                row_above.append(local_spearman(ytest, yhat, threshold, above=True))
                row_below.append(local_spearman(ytest, yhat, threshold, above=False))
            corr_above.append(row_above)
            corr_below.append(row_below)
        
        all_upper = np.array(corr_above)  # (n_trials, n_tasks)
        all_lower = np.array(corr_below)
        
        sensitive_data = all_upper if censor_region == 'above' else all_lower
        
        # subsample xnoise to 11 levels
        if method_name == 'Feature Noise':
            sensitive_data = sensitive_data[:, xnoise_indices]
        
        method_sensitive[method_name] = sensitive_data  # (n_trials, 11)
    
    # compare across 11 matched intensity levels
    results = []
    n_levels = 11
    
    for level_idx in range(1, n_levels):  # skip baseline (index 0)
        for method_a, method_b in [('Feature Noise', 'Label Noise'), 
                                    ('Feature Noise', 'Omission'),
                                    ('Label Noise', 'Omission')]:
            
            if method_a not in method_sensitive or method_b not in method_sensitive:
                continue
            
            baseline_a = method_sensitive[method_a][:, 0]
            baseline_b = method_sensitive[method_b][:, 0]
            
            current_a = method_sensitive[method_a][:, level_idx]
            current_b = method_sensitive[method_b][:, level_idx]
            
            deg_a = (baseline_a - current_a) / np.abs(baseline_a) * 100
            deg_b = (baseline_b - current_b) / np.abs(baseline_b) * 100
            
            try:
                _, p_value = wilcoxon(deg_a, deg_b, alternative='greater')
            except ValueError:
                p_value = 1.0
            
            sig_marker = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
            
            results.append({
                'Level_Index': level_idx,
                'Comparison': f"{method_a} vs {method_b}",
                'Avg_Deg_A': f"{np.mean(deg_a):.1f}%",
                'Avg_Deg_B': f"{np.mean(deg_b):.1f}%",
                'A_Superior': p_value < 0.05,
                'p_value': p_value,
                'p_formatted': f"{p_value:.3f}{sig_marker}",
            })
    
    return pd.DataFrame(results)

In [ ]:
# with 10% sensitive data
censor_split= 0.1

history_files = [
    f'./all_results/mlp_xnoise_results_split{censor_split}_{censor_region}/history.json',
    f'./all_results/mlp_ynoise_results_split{censor_split}_{censor_region}/history.json', 
    f'./all_results/mlp_omit_results_split{censor_split}_{censor_region}/history.json'
]

method_names = ['Feature Noise', 'Label Noise', 'Omission']

df = test_method_effectiveness_mlp(
    history_files, 
    method_names, 
    censor_region=censor_region,
    censor_split=censor_split
)

df

In [ ]:
# with 50% sensitive data
censor_split= 0.5

history_files = [
    f'./all_results/mlp_xnoise_results_split{censor_split}_{censor_region}/history.json',
    f'./all_results/mlp_ynoise_results_split{censor_split}_{censor_region}/history.json', 
    f'./all_results/mlp_omit_results_split{censor_split}_{censor_region}/history.json'
]

method_names = ['Feature Noise', 'Label Noise', 'Omission']

df = test_method_effectiveness_mlp(
    history_files, 
    method_names, 
    censor_region=censor_region,
    censor_split=censor_split
)

df

In [ ]:
# with 90% sensitive data
censor_split= 0.9

history_files = [
    f'./all_results/mlp_xnoise_results_split{censor_split}_{censor_region}/history.json',
    f'./all_results/mlp_ynoise_results_split{censor_split}_{censor_region}/history.json', 
    f'./all_results/mlp_omit_results_split{censor_split}_{censor_region}/history.json'
]

method_names = ['Feature Noise', 'Label Noise', 'Omission']

df = test_method_effectiveness_mlp(
    history_files, 
    method_names, 
    censor_region=censor_region,
    censor_split=censor_split
)

df